# NLP Practical 1 — Finite State Automata, RTN & ATN

Run in **Google Colab**. Cells with `pip install` only need to run once per session.

**Covers:** FSA formalism (Q, Σ, δ, q0, F) for English + Hindi morphology, a Recursive Transition Network (RTN) for nested noun phrases, an Augmented Transition Network (ATN) for subject-verb agreement, and a spelling-error detector built on top of the FSA.


In [ ]:
!pip install graphviz -q


In [ ]:
# ============================================================
# PART A: Finite State Automaton (FSA) for English + Hindi morphology
# FSA = (Q, Sigma, delta, q0, F)
# ============================================================

class FSA:
    def __init__(self, states, alphabet, transitions, start, accept):
        self.Q = states
        self.Sigma = alphabet
        self.delta = transitions   # dict[(state, symbol)] -> state
        self.q0 = start
        self.F = accept

    def run(self, string):
        state = self.q0
        path = [state]
        for ch in string:
            key = (state, ch)
            if key not in self.delta:
                return False, path
            state = self.delta[key]
            path.append(state)
        return state in self.F, path

# --- English plural-morphology FSA: recognizes stem + 's' / 'es' ---
# Toy alphabet: any lowercase letter for the stem, then 's'/'es' suffix
english_plural_fsa = FSA(
    states={"q_stem", "q_s", "q_e"},
    alphabet=set("abcdefghijklmnopqrstuvwxyz"),
    transitions={
        **{("q_stem", ch): "q_stem" for ch in "abcdefghijklmnopqrstuvwxyz"},
        ("q_stem", "s"): "q_s",
        ("q_stem", "e"): "q_e",
        ("q_e", "s"): "q_s",
    },
    start="q_stem",
    accept={"q_s"}
)

for word in ["cats", "boxes", "cat", "dog"]:
    accepted, path = english_plural_fsa.run(word)
    print(f"{word:8s} -> accepted={accepted}")

# --- Simple Hindi (transliterated) plural-marker FSA: stem + 'on'/'en' ---
hindi_plural_fsa = FSA(
    states={"q_stem", "q_o", "q_n"},
    alphabet=set("abcdefghijklmnopqrstuvwxyz"),
    transitions={
        **{("q_stem", ch): "q_stem" for ch in "abcdefghijklmnopqrstuvwxyz"},
        ("q_stem", "o"): "q_o",
        ("q_o", "n"): "q_n",
    },
    start="q_stem",
    accept={"q_n"}
)

for word in ["ladkon", "kitaben", "ladka"]:
    accepted, path = hindi_plural_fsa.run(word)
    print(f"{word:10s} -> accepted={accepted}")

print("\nTakeaway: modern subword tokenizers (BPE / WordPiece / SentencePiece) are")
print("mathematically finite-state segmenters -- this is the same idea at industrial scale.")


In [ ]:
# ============================================================
# PART B: Recursive Transition Network (RTN) for Noun Phrases
# NP -> Det (Adj)* N (PP)*   where PP -> Prep NP   (recursive!)
# ============================================================

DET = {"the", "a", "an"}
ADJ = {"big", "small", "red", "old", "quiet"}
NOUN = {"boy", "girl", "book", "table", "garden", "school"}
PREP = {"in", "on", "near", "with"}

def parse_NP(tokens, i):
    # Recursive-descent parser acting as an RTN. Returns (tree, next_index) or None.
    if i >= len(tokens) or tokens[i] not in DET:
        return None
    det = tokens[i]; i += 1
    adjs = []
    while i < len(tokens) and tokens[i] in ADJ:
        adjs.append(tokens[i]); i += 1
    if i >= len(tokens) or tokens[i] not in NOUN:
        return None
    noun = tokens[i]; i += 1
    tree = {"NP": [det] + adjs + [noun]}
    # optional recursive PP attachment: PP -> Prep NP
    while i < len(tokens) and tokens[i] in PREP:
        prep = tokens[i]; i += 1
        sub = parse_NP(tokens, i)
        if sub is None:
            break
        sub_tree, i = sub
        tree.setdefault("PP", []).append({"prep": prep, "NP": sub_tree})
    return tree, i

sentence = "the old book on the small table in the garden".split()
result = parse_NP(sentence, 0)
print("Recursive RTN parse of NP with nested PPs:")
import pprint; pprint.pprint(result[0] if result else "REJECTED")


In [ ]:
# ============================================================
# PART C: Augmented Transition Network (ATN) with subject-verb agreement
# Registers hold number/person features and gate transitions.
# ============================================================

VERB_FORMS = {
    "boy": {"sing": "studies"}, "boys": {"plur": "study"},
    "girl": {"sing": "studies"}, "girls": {"plur": "study"},
}

def atn_check_agreement(subject, verb):
    number = "plur" if subject.endswith("s") else "sing"
    expected = VERB_FORMS.get(subject, {}).get(number)
    register = {"subject": subject, "number": number, "verb_seen": verb}
    ok = (expected == verb)
    register["expected_verb"] = expected
    register["agreement_ok"] = ok
    return register

for subj, vb in [("boy", "studies"), ("boys", "studies"), ("girls", "study")]:
    print(atn_check_agreement(subj, vb))

print("\nIndustry link: ATN-style register/feature checking is the ancestor of the")
print("feature-unification used in modern grammar checkers (e.g. Grammarly's rule layer).")


In [ ]:
# ============================================================
# PART D: Spelling-error detection using the FSA dictionary
# A word is flagged as a possible error if the FSA does NOT accept it.
# ============================================================

dictionary_words = ["cats", "dogs", "boxes", "houses"]
test_words = ["cats", "catz", "boxs", "houses", "hous"]

def is_valid_plural(word, fsa):
    accepted, _ = fsa.run(word)
    return accepted

for w in test_words:
    flag = "OK" if is_valid_plural(w, english_plural_fsa) else "POSSIBLE SPELLING ERROR"
    print(f"{w:8s} -> {flag}")
